In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from transformers import CamembertModel, AutoTokenizer, CLIPVisionModel, CLIPProcessor
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

BATCH_SIZE = 16
ACCUM_STEPS = 2
MAX_TEXT_LEN = 128
NUM_WORKERS = 0
EARLY_STOPPING_PATIENCE = 6
RESUME = True
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.1
PROJECTION_DIM = 512

CAMEMBERT_ID = "camembert-base"
CLIP_ID = "openai/clip-vit-base-patch32"

STAGE_CONFIGS = [
    {
        "name": "stage1_head_only",
        "epochs": 3,
        "text_lr": 0.0,
        "vision_lr": 0.0,
        "projection_lr": 5e-4,
        "gate_lr": 5e-4,
        "classifier_lr": 5e-4
    },
    {
        "name": "stage2_partial_unfreeze",
        "epochs": 4,
        "text_lr": 2e-6,
        "vision_lr": 2e-6,
        "projection_lr": 1e-4,
        "gate_lr": 1e-4,
        "classifier_lr": 1e-4
    },
    {
        "name": "stage3_full_unfreeze",
        "epochs": 8,
        "text_lr": 1e-6,
        "vision_lr": 1e-6,
        "projection_lr": 5e-5,
        "gate_lr": 5e-5,
        "classifier_lr": 5e-5
    }
]

BASE_DIR = Path.cwd().parent.parent.parent.parent
DATA_RAW = BASE_DIR / "data" / "raw"
IMAGE_DIR = DATA_RAW / "images" / "image_train"
SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"

RUN_NAME = "multimodal_camembert_clip_proj_gate_aug_resume"
OUTPUT_DIR = BASE_DIR / "outputs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LAST_CHECKPOINT_PATH = OUTPUT_DIR / "last_checkpoint.pt"
BEST_CHECKPOINT_PATH = OUTPUT_DIR / "best_checkpoint.pt"
BEST_MODEL_PATH = OUTPUT_DIR / "best_model_state_dict.pt"
BEST_VAL_LOGITS_PATH = OUTPUT_DIR / "best_val_logits.npy"
BEST_VAL_GATE_PATH = OUTPUT_DIR / "best_val_gate_weights.npy"
BEST_VAL_PRED_PATH = OUTPUT_DIR / "best_val_predictions.csv"
HISTORY_CSV_PATH = OUTPUT_DIR / "training_history.csv"
PLOT_PATH = OUTPUT_DIR / "metrics_curves.png"
BEST_REPORT_PATH = OUTPUT_DIR / "best_classification_report.txt"
BEST_REPORT_CSV_PATH = OUTPUT_DIR / "best_classification_report.csv"
BEST_METADATA_PATH = OUTPUT_DIR / "best_metadata.json"

train_df = pd.read_csv(SPLIT_DIR / "train_split.csv")
val_df = pd.read_csv(SPLIT_DIR / "val_split.csv")

with open(SPLIT_DIR / "label2id.json", "r", encoding="utf-8") as f:
    raw_label2id = json.load(f)
    label2id = {int(k): int(v) for k, v in raw_label2id.items()}

id2label = {v: k for k, v in label2id.items()}
NUM_CLASSES = len(label2id)

tokenizer = AutoTokenizer.from_pretrained(CAMEMBERT_ID)
processor = CLIPProcessor.from_pretrained(CLIP_ID)

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.05),
    transforms.RandomAffine(degrees=5, translate=(0.03, 0.03), scale=(0.97, 1.03))
])

val_transforms = None


class RakutenDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = IMAGE_DIR / f"image_{row['imageid']}_product_{row['productid']}.jpg"
        image = Image.open(img_path).convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        designation = "" if pd.isna(row.get("designation", "")) else str(row.get("designation", ""))
        description = "" if pd.isna(row.get("description", "")) else str(row.get("description", ""))
        text = f"{designation} {description}".strip()

        txt = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=MAX_TEXT_LEN,
            return_tensors="pt"
        )

        img = processor(images=image, return_tensors="pt")

        return {
            "input_ids": txt["input_ids"].squeeze(0),
            "attention_mask": txt["attention_mask"].squeeze(0),
            "pixel_values": img["pixel_values"].squeeze(0),
            "labels": torch.tensor(label2id[int(row["prdtypecode"])], dtype=torch.long),
            "imageid": int(row["imageid"]),
            "productid": int(row["productid"]),
            "prdtypecode": int(row["prdtypecode"])
        }


train_loader = DataLoader(
    RakutenDataset(train_df, transform=train_transforms),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    RakutenDataset(val_df, transform=val_transforms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)


class FusionModel(nn.Module):
    def __init__(self, num_classes, projection_dim=512):
        super().__init__()

        self.text_backbone = CamembertModel.from_pretrained(CAMEMBERT_ID)
        self.vision_backbone = CLIPVisionModel.from_pretrained(CLIP_ID)

        self.text_projection = nn.Sequential(
            nn.Linear(768, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.GELU()
        )

        self.vision_projection = nn.Sequential(
            nn.Linear(768, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.GELU()
        )

        self.gate = nn.Sequential(
            nn.Linear(projection_dim * 2, projection_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(projection_dim, 2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(projection_dim, projection_dim),
            nn.BatchNorm1d(projection_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(projection_dim, num_classes)
        )

        self.debug_done = False

    def forward(self, input_ids, attention_mask, pixel_values):
        text_outputs = self.text_backbone(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        vision_outputs = self.vision_backbone(
            pixel_values=pixel_values
        )

        t_feat = text_outputs.last_hidden_state[:, 0]
        v_feat = vision_outputs.pooler_output

        t_proj = self.text_projection(t_feat)
        v_proj = self.vision_projection(v_feat)

        t_proj = F.normalize(t_proj, p=2, dim=1)
        v_proj = F.normalize(v_proj, p=2, dim=1)

        gate_input = torch.cat([t_proj, v_proj], dim=1)
        gate_logits = self.gate(gate_input)
        gate_weights = torch.softmax(gate_logits, dim=1)

        w_text = gate_weights[:, 0].unsqueeze(1)
        w_image = gate_weights[:, 1].unsqueeze(1)

        fused = (w_text * t_proj) + (w_image * v_proj)
        logits = self.classifier(fused)

        if not self.debug_done:
            print("text raw shape:", t_feat.shape)
            print("vision raw shape:", v_feat.shape)
            print("text proj shape:", t_proj.shape)
            print("vision proj shape:", v_proj.shape)
            print("gate input shape:", gate_input.shape)
            print("gate weights shape:", gate_weights.shape)
            print("fused shape:", fused.shape)
            print("logits shape:", logits.shape)
            self.debug_done = True

        return logits, gate_weights


model = FusionModel(num_classes=NUM_CLASSES, projection_dim=PROJECTION_DIM).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)


def freeze_module(module):
    for p in module.parameters():
        p.requires_grad = False


def unfreeze_module(module):
    for p in module.parameters():
        p.requires_grad = True


def freeze_all_backbones(model):
    freeze_module(model.text_backbone)
    freeze_module(model.vision_backbone)


def unfreeze_last_camembert_layers(model, layer_indices=(10, 11)):
    for name, p in model.text_backbone.named_parameters():
        if any(f"encoder.layer.{idx}." in name for idx in layer_indices):
            p.requires_grad = True


def unfreeze_last_clip_layers(model, layer_indices=(10, 11)):
    for name, p in model.vision_backbone.named_parameters():
        if any(f"vision_model.encoder.layers.{idx}." in name for idx in layer_indices):
            p.requires_grad = True
        elif "vision_model.post_layernorm" in name:
            p.requires_grad = True


def apply_stage_freezing(model, stage_name):
    unfreeze_module(model.text_projection)
    unfreeze_module(model.vision_projection)
    unfreeze_module(model.gate)
    unfreeze_module(model.classifier)

    if stage_name == "stage1_head_only":
        freeze_all_backbones(model)
    elif stage_name == "stage2_partial_unfreeze":
        freeze_all_backbones(model)
        unfreeze_last_camembert_layers(model, layer_indices=(10, 11))
        unfreeze_last_clip_layers(model, layer_indices=(10, 11))
    elif stage_name == "stage3_full_unfreeze":
        unfreeze_module(model)
    else:
        raise ValueError(f"Unknown stage name: {stage_name}")


def build_optimizer(model, text_lr, vision_lr, projection_lr, gate_lr, classifier_lr):
    text_params = []
    vision_params = []
    projection_params = []
    gate_params = []
    classifier_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        if name.startswith("text_backbone."):
            text_params.append(p)
        elif name.startswith("vision_backbone."):
            vision_params.append(p)
        elif name.startswith("text_projection.") or name.startswith("vision_projection."):
            projection_params.append(p)
        elif name.startswith("gate."):
            gate_params.append(p)
        elif name.startswith("classifier."):
            classifier_params.append(p)

    param_groups = []

    if text_params and text_lr > 0:
        param_groups.append({"params": text_params, "lr": text_lr})

    if vision_params and vision_lr > 0:
        param_groups.append({"params": vision_params, "lr": vision_lr})

    if projection_params and projection_lr > 0:
        param_groups.append({"params": projection_params, "lr": projection_lr})

    if gate_params and gate_lr > 0:
        param_groups.append({"params": gate_params, "lr": gate_lr})

    if classifier_params and classifier_lr > 0:
        param_groups.append({"params": classifier_params, "lr": classifier_lr})

    return torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)


def build_advanced_optimizer(model, stage_cfg):
    optimizer = build_optimizer(
        model=model,
        text_lr=stage_cfg["text_lr"],
        vision_lr=stage_cfg["vision_lr"],
        projection_lr=stage_cfg["projection_lr"],
        gate_lr=stage_cfg["gate_lr"],
        classifier_lr=stage_cfg["classifier_lr"]
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2,
        threshold=1e-3,
        threshold_mode="rel"
    )
    return optimizer, scheduler


def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def run_epoch(model, loader, optimizer=None, train=True):
    if train:
        model.train()
        optimizer.zero_grad(set_to_none=True)
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_true = []
    all_logits = []
    all_gate_weights = []
    all_rows = []

    for step, batch in enumerate(tqdm(loader, leave=False), start=1):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        pixel_values = batch["pixel_values"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.set_grad_enabled(train):
            logits, gate_weights = model(input_ids, attention_mask, pixel_values)
            loss = criterion(logits, labels)

            if train:
                (loss / ACCUM_STEPS).backward()

                if step % ACCUM_STEPS == 0:
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * labels.size(0)

        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        true_np = labels.detach().cpu().numpy()

        all_preds.extend(preds)
        all_true.extend(true_np)

        if not train:
            logits_np = logits.detach().cpu().numpy()
            gate_np = gate_weights.detach().cpu().numpy()

            all_logits.append(logits_np)
            all_gate_weights.append(gate_np)

            imageids = batch["imageid"]
            productids = batch["productid"]
            prdtypecodes = batch["prdtypecode"]

            for idx in range(len(preds)):
                pred_label_id = int(preds[idx])
                true_label_id = int(true_np[idx])

                row = {
                    "imageid": int(imageids[idx]),
                    "productid": int(productids[idx]),
                    "true_prdtypecode": int(prdtypecodes[idx]),
                    "true_label_id": true_label_id,
                    "pred_label_id": pred_label_id,
                    "pred_prdtypecode": int(id2label[pred_label_id]),
                    "text_weight": float(gate_np[idx, 0]),
                    "image_weight": float(gate_np[idx, 1])
                }

                for class_idx in range(logits_np.shape[1]):
                    row[f"logit_{class_idx}"] = float(logits_np[idx, class_idx])

                all_rows.append(row)

    if train and (len(loader) % ACCUM_STEPS != 0):
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    epoch_loss = total_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_true, all_preds)
    epoch_f1 = f1_score(all_true, all_preds, average="macro")

    logits_out = np.concatenate(all_logits, axis=0) if (not train and all_logits) else None
    gate_out = np.concatenate(all_gate_weights, axis=0) if (not train and all_gate_weights) else None
    pred_df = pd.DataFrame(all_rows) if not train else None

    return epoch_loss, epoch_acc, epoch_f1, logits_out, gate_out, all_true, all_preds, pred_df


def save_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    history,
    global_epoch,
    stage_idx,
    local_epoch,
    best_f1,
    best_epoch_global,
    best_stage_name,
    epochs_without_improvement
):
    payload = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
        "history": history,
        "global_epoch": global_epoch,
        "stage_idx": stage_idx,
        "local_epoch": local_epoch,
        "best_f1": best_f1,
        "best_epoch_global": best_epoch_global,
        "best_stage_name": best_stage_name,
        "epochs_without_improvement": epochs_without_improvement,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "accum_steps": ACCUM_STEPS,
        "max_text_len": MAX_TEXT_LEN,
        "camembert_id": CAMEMBERT_ID,
        "clip_id": CLIP_ID,
        "stage_configs": STAGE_CONFIGS
    }
    torch.save(payload, path)


def load_checkpoint(path):
    return torch.load(path, map_location=DEVICE)


def plot_history(history, output_path):
    if len(history) == 0:
        return

    df = pd.DataFrame(history)

    plt.figure(figsize=(10, 6))
    plt.plot(df["global_epoch"], df["train_acc"], label="Train Accuracy")
    plt.plot(df["global_epoch"], df["val_acc"], label="Validation Accuracy")
    plt.plot(df["global_epoch"], df["train_macro_f1"], label="Train Macro F1")
    plt.plot(df["global_epoch"], df["val_macro_f1"], label="Validation Macro F1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.title("Training Metrics")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()


history = []
best_f1 = -1.0
best_epoch_global = -1
best_stage_name = None
epochs_without_improvement = 0
global_epoch = 0
start_stage_idx = 0
resume_local_epoch = 0

if RESUME and LAST_CHECKPOINT_PATH.exists():
    print(f"Resuming from checkpoint: {LAST_CHECKPOINT_PATH}")
    ckpt = load_checkpoint(LAST_CHECKPOINT_PATH)

    model.load_state_dict(ckpt["model_state_dict"])
    history = ckpt["history"]
    global_epoch = ckpt["global_epoch"]
    start_stage_idx = ckpt["stage_idx"]
    resume_local_epoch = ckpt["local_epoch"]
    best_f1 = ckpt["best_f1"]
    best_epoch_global = ckpt["best_epoch_global"]
    best_stage_name = ckpt["best_stage_name"]
    epochs_without_improvement = ckpt["epochs_without_improvement"]
else:
    print("Starting training from scratch")

stop_training = False

for stage_idx in range(start_stage_idx, len(STAGE_CONFIGS)):
    stage_cfg = STAGE_CONFIGS[stage_idx]
    stage_name = stage_cfg["name"]
    stage_epochs = stage_cfg["epochs"]

    print("\n" + "=" * 80)
    print(f"Starting {stage_name}")
    print("=" * 80)

    apply_stage_freezing(model, stage_name)
    optimizer, scheduler = build_advanced_optimizer(model, stage_cfg)

    if RESUME and LAST_CHECKPOINT_PATH.exists() and stage_idx == start_stage_idx:
        ckpt = load_checkpoint(LAST_CHECKPOINT_PATH)

        if ckpt["optimizer_state_dict"] is not None:
            try:
                optimizer.load_state_dict(ckpt["optimizer_state_dict"])
                print("Optimizer state loaded from checkpoint")
            except Exception as e:
                print(f"Optimizer state could not be loaded exactly: {e}")

        if ckpt.get("scheduler_state_dict") is not None:
            try:
                scheduler.load_state_dict(ckpt["scheduler_state_dict"])
                print("Scheduler state loaded from checkpoint")
            except Exception as e:
                print(f"Scheduler state could not be loaded exactly: {e}")

    trainable, total = count_trainable_params(model)
    print(f"Trainable params: {trainable:,} / {total:,}")
    print(f"Text LR: {stage_cfg['text_lr']}")
    print(f"Vision LR: {stage_cfg['vision_lr']}")
    print(f"Projection LR: {stage_cfg['projection_lr']}")
    print(f"Gate LR: {stage_cfg['gate_lr']}")
    print(f"Classifier LR: {stage_cfg['classifier_lr']}")

    stage_start_epoch = resume_local_epoch + 1 if stage_idx == start_stage_idx else 1

    for local_epoch in range(stage_start_epoch, stage_epochs + 1):
        global_epoch += 1

        train_loss, train_acc, train_f1, _, _, _, _, _ = run_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            train=True
        )

        val_loss, val_acc, val_f1, val_logits, val_gate, y_true, y_pred, val_pred_df = run_epoch(
            model=model,
            loader=val_loader,
            optimizer=None,
            train=False
        )

        scheduler.step(val_f1)

        mean_text_weight = float(val_gate[:, 0].mean()) if val_gate is not None else None
        mean_image_weight = float(val_gate[:, 1].mean()) if val_gate is not None else None

        print(
            f"Epoch {global_epoch:02d} | "
            f"Stage={stage_name} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.4f} | "
            f"train_f1={train_f1:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"val_acc={val_acc:.4f} | "
            f"val_f1={val_f1:.4f} | "
            f"mean_text_w={mean_text_weight:.4f} | "
            f"mean_image_w={mean_image_weight:.4f}"
        )

        history.append({
            "global_epoch": global_epoch,
            "stage_idx": stage_idx,
            "stage": stage_name,
            "stage_epoch": local_epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_macro_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_macro_f1": val_f1,
            "mean_text_weight": mean_text_weight,
            "mean_image_weight": mean_image_weight,
            "text_lr": stage_cfg["text_lr"],
            "vision_lr": stage_cfg["vision_lr"],
            "projection_lr": stage_cfg["projection_lr"],
            "gate_lr": stage_cfg["gate_lr"],
            "classifier_lr": stage_cfg["classifier_lr"]
        })

        pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)
        plot_history(history, PLOT_PATH)

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch_global = global_epoch
            best_stage_name = stage_name
            epochs_without_improvement = 0

            torch.save(model.state_dict(), BEST_MODEL_PATH)
            np.save(BEST_VAL_LOGITS_PATH, val_logits)
            np.save(BEST_VAL_GATE_PATH, val_gate)
            val_pred_df.to_csv(BEST_VAL_PRED_PATH, index=False)

            report_text = classification_report(y_true, y_pred, digits=4)
            with open(BEST_REPORT_PATH, "w", encoding="utf-8") as f:
                f.write(report_text)

            report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
            pd.DataFrame(report_dict).transpose().to_csv(BEST_REPORT_CSV_PATH)

            best_metadata = {
                "best_val_macro_f1": float(best_f1),
                "best_epoch_global": int(best_epoch_global),
                "best_stage_name": best_stage_name,
                "num_classes": int(NUM_CLASSES),
                "batch_size": int(BATCH_SIZE),
                "accum_steps": int(ACCUM_STEPS),
                "max_text_len": int(MAX_TEXT_LEN),
                "label_smoothing": float(LABEL_SMOOTHING),
                "projection_dim": int(PROJECTION_DIM),
                "camembert_id": CAMEMBERT_ID,
                "clip_id": CLIP_ID,
                "device": str(DEVICE),
                "best_val_logits_path": str(BEST_VAL_LOGITS_PATH),
                "best_val_gate_weights_path": str(BEST_VAL_GATE_PATH),
                "best_val_predictions_path": str(BEST_VAL_PRED_PATH)
            }

            with open(BEST_METADATA_PATH, "w", encoding="utf-8") as f:
                json.dump(best_metadata, f, indent=2, ensure_ascii=False)

            save_checkpoint(
                path=BEST_CHECKPOINT_PATH,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                history=history,
                global_epoch=global_epoch,
                stage_idx=stage_idx,
                local_epoch=local_epoch,
                best_f1=best_f1,
                best_epoch_global=best_epoch_global,
                best_stage_name=best_stage_name,
                epochs_without_improvement=epochs_without_improvement
            )

            print(f"  -> New best model saved. val_f1={val_f1:.4f}")

        else:
            epochs_without_improvement += 1
            print(f"  -> No improvement for {epochs_without_improvement} epoch(s)")

        save_checkpoint(
            path=LAST_CHECKPOINT_PATH,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            history=history,
            global_epoch=global_epoch,
            stage_idx=stage_idx,
            local_epoch=local_epoch,
            best_f1=best_f1,
            best_epoch_global=best_epoch_global,
            best_stage_name=best_stage_name,
            epochs_without_improvement=epochs_without_improvement
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("\nEarly stopping triggered.")
            stop_training = True
            break

    resume_local_epoch = 0

    if stop_training:
        break

print("\nTraining finished.")
print(f"Best global epoch: {best_epoch_global}")
print(f"Best stage: {best_stage_name}")
print(f"Best val macro F1: {best_f1:.4f}")
print(f"History CSV: {HISTORY_CSV_PATH}")
print(f"Plot: {PLOT_PATH}")
print(f"Best model weights: {BEST_MODEL_PATH}")
print(f"Best val logits: {BEST_VAL_LOGITS_PATH}")
print(f"Best val gate weights: {BEST_VAL_GATE_PATH}")
print(f"Best val predictions: {BEST_VAL_PRED_PATH}")
print(f"Last checkpoint: {LAST_CHECKPOINT_PATH}")
print(f"Best checkpoint: {BEST_CHECKPOINT_PATH}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# PATH
BASE_DIR = Path.cwd().parent.parent.parent.parent
RUN_NAME = "multimodal_camembert_clip_proj_gate_aug_resume"

HISTORY_CSV_PATH = BASE_DIR / "outputs" / RUN_NAME / "training_history.csv"

# LOAD
df = pd.read_csv(HISTORY_CSV_PATH)

print(df.head())

# =========================
# ACCURACY
# =========================
plt.figure(figsize=(10,6))
plt.plot(df["global_epoch"], df["train_acc"], label="Train Accuracy")
plt.plot(df["global_epoch"], df["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy over Epochs")
plt.legend()
plt.grid(False)
plt.show()

# =========================
# F1 SCORE
# =========================
plt.figure(figsize=(10,6))
plt.plot(df["global_epoch"], df["train_macro_f1"], label="Train Macro F1")
plt.plot(df["global_epoch"], df["val_macro_f1"], label="Validation Macro F1")
plt.xlabel("Epoch")
plt.ylabel("Macro F1")
plt.title("Macro F1 over Epochs")
plt.legend()
plt.grid(False)
plt.show()

# =========================
# LOSS
# =========================
plt.figure(figsize=(10,6))
plt.plot(df["global_epoch"], df["train_loss"], label="Train Loss")
plt.plot(df["global_epoch"], df["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss over Epochs")
plt.legend()
plt.grid(False)
plt.show()

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

BASE_DIR = Path.cwd().parent.parent.parent.parent

IMAGE_RUN_NAME = "multimodal_camembert_clip_proj_gate_aug_resume"

SPLIT_DIR = BASE_DIR / "outputs" / "image_modeling"
IMAGE_RUN_DIR = BASE_DIR / "outputs" / IMAGE_RUN_NAME
TEXT_RUN_DIR = BASE_DIR / "outputs"

VAL_SPLIT_PATH = SPLIT_DIR / "val_split.csv"
LABEL2ID_PATH = SPLIT_DIR / "label2id.json"

IMAGE_LOGITS_PATH = IMAGE_RUN_DIR / "best_val_logits.npy"
TEXT_LOGITS_PATH = TEXT_RUN_DIR / "y_logits_camembert_run2_epochs3.npy"

val_df = pd.read_csv(VAL_SPLIT_PATH)

with open(LABEL2ID_PATH, "r", encoding="utf-8") as f:
    raw_label2id = json.load(f)
    label2id = {int(k): int(v) for k, v in raw_label2id.items()}

y_true = val_df["prdtypecode"].map(label2id).to_numpy()

image_logits = np.load(IMAGE_LOGITS_PATH)
text_logits = np.load(TEXT_LOGITS_PATH)

print("image_logits shape:", image_logits.shape)
print("text_logits shape:", text_logits.shape)
print("y_true shape:", y_true.shape)

if len(image_logits) != len(text_logits) or len(image_logits) != len(y_true):
    raise ValueError("Logits ve y_true satır sayıları aynı değil.")

X = np.concatenate([image_logits, text_logits], axis=1)

X_meta_train, X_meta_val, y_meta_train, y_meta_val = train_test_split(
    X,
    y_true,
    test_size=0.3,
    random_state=42,
    stratify=y_true
)

meta_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        C=1.0
    ))
])

meta_model.fit(X_meta_train, y_meta_train)

y_pred = meta_model.predict(X_meta_val)
y_proba = meta_model.predict_proba(X_meta_val)

acc = accuracy_score(y_meta_val, y_pred)
macro_f1 = f1_score(y_meta_val, y_pred, average="macro")
weighted_f1 = f1_score(y_meta_val, y_pred, average="weighted")

print("STACKING LOGIT FUSION RESULTS")
print("Accuracy :", round(acc, 6))
print("Macro F1 :", round(macro_f1, 6))
print("Weighted F1 :", round(weighted_f1, 6))
print()
print(classification_report(y_meta_val, y_pred, digits=4))

OUTPUT_DIR = BASE_DIR / "outputs" / "clip_aug_late_fusion"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUTPUT_DIR / "X_meta_train.npy", X_meta_train)
np.save(OUTPUT_DIR / "X_meta_val.npy", X_meta_val)
np.save(OUTPUT_DIR / "y_meta_train.npy", y_meta_train)
np.save(OUTPUT_DIR / "y_meta_val.npy", y_meta_val)
np.save(OUTPUT_DIR / "y_pred.npy", y_pred)
np.save(OUTPUT_DIR / "y_proba.npy", y_proba)

report_df = pd.DataFrame(classification_report(
    y_meta_val,
    y_pred,
    output_dict=True,
    zero_division=0
)).transpose()
report_df.to_csv(OUTPUT_DIR / "classification_report.csv")

pred_df = pd.DataFrame({
    "true_label_id": y_meta_val,
    "pred_label_id": y_pred,
    "max_proba": y_proba.max(axis=1)
})
pred_df.to_csv(OUTPUT_DIR / "meta_val_predictions.csv", index=False)

print("Saved to:", OUTPUT_DIR)